# Part 1: STL Decomposition + Time-Series Cross-Validation
**⏱ This section takes approximately 30 minutes.**

---

## Scenario: Tuesday — Separate the Signal from the Cycle from the Noise

Yesterday Sarah saw the data has TREND + ANNUAL SEASONALITY + WEEKLY SEASONALITY + NOISE — all overlapping. Today she separates them with STL decomposition, then learns the right way to cross-validate time-series models.

**Why decompose?** Because each component has different shape, scale, and predictability. By separating them we (a) understand what's driving the series, (b) can forecast each piece with the right tool, and (c) can spot anomalies in the residual that we couldn't see in the raw data.

**Why specialised cross-validation?** Because random k-fold puts future data in earlier training folds — leakage. `TimeSeriesSplit` enforces "always train on the past, test on the future."

**By the end of this notebook you will be able to:**
- Run STL decomposition on a daily time series
- Read trend / seasonal / residual components
- Set up time-series cross-validation correctly
- Compute MAE / RMSE / MAPE for a forecast

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

from statsmodels.tsa.seasonal import STL
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (13, 4.5)

print("✅ Libraries loaded — STL, TimeSeriesSplit ready")

## Step 1 — Load + DatetimeIndex

In [ ]:
df = pd.read_csv("data/northstar_daily_revenue.csv", parse_dates=["date"])
df = df.set_index("date")
y = df["revenue_gbp"]   # the target series — one revenue value per day

print(f"Loaded {len(y)} daily revenue observations")
print(f"Index type: {type(y.index).__name__}")
print(f"Frequency: {y.index.inferred_freq}")

## Step 2 — STL with period = 7 (weekly seasonality)

STL needs you to specify the SEASONAL PERIOD. Daily data with weekly seasonality → period=7.

(STL can only handle ONE seasonality. Our data has BOTH weekly and annual — we model the weekly one explicitly here. The annual pattern shows up in the trend component.)

In [ ]:
stl = STL(y, period=7, robust=True).fit()

# Plot all four panels: observed, trend, seasonal, residual
fig = stl.plot()
fig.set_size_inches(13, 8)
for ax in fig.axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 💡 What you should notice

- **Observed** (top panel): the raw series — what we plotted in NB 01.
- **Trend**: smooth upward curve with the annual cycle visible. Because STL was given period=7, the annual cycle wasn't allocated to "seasonal" and ends up in the trend.
- **Seasonal**: the 7-day weekly cycle, repeating consistently across the whole series.
- **Residual**: what's left — should look like noise. Spikes in the residual are days that surprised the model.

**The decomposition adds up:** `observed = trend + seasonal + residual`. Let's verify.

In [ ]:
# Sanity check
reconstructed = stl.trend + stl.seasonal + stl.resid
max_abs_diff = (reconstructed - y).abs().max()
print(f"Max |reconstructed - observed| = {max_abs_diff:.6f}")
print("→ Essentially zero. The components add up exactly to the original series.")

## Step 3 — Inspect the seasonal component

Pull out the weekly seasonal pattern. What's the average lift on each day of the week?

In [ ]:
seasonal_by_dow = (
    pd.DataFrame({"seasonal": stl.seasonal, "dow": stl.seasonal.index.dayofweek})
    .groupby("dow")["seasonal"].mean()
    .round(0)
)
seasonal_by_dow.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

fig, ax = plt.subplots(figsize=(9, 4))
seasonal_by_dow.plot(kind="bar", ax=ax, color=["steelblue"]*5 + ["coral"]*2, edgecolor="white")
ax.set_ylabel("Mean seasonal effect (£)")
ax.set_xlabel("")
ax.set_title("Weekly seasonality — average effect per day of week")
ax.axhline(0, color="black", linewidth=0.5)
for i, v in enumerate(seasonal_by_dow):
    ax.text(i, v + (30 if v > 0 else -30), f"£{v:+.0f}",
            ha="center", va="bottom" if v > 0 else "top", fontsize=10, fontweight="bold")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("Weekend effect:", f"+£{seasonal_by_dow[['Sat','Sun']].mean():.0f}/day")
print("Weekday effect:", f"£{seasonal_by_dow[['Mon','Tue','Wed','Thu']].mean():.0f}/day")

## Step 4 — Look at the trend by month

The trend captures slow-moving structure. With period=7, the trend retains BOTH the long-term growth and the annual cycle. Plotting it makes the annual pattern visible cleanly.

In [ ]:
# Group trend by month
trend_monthly = stl.trend.resample("ME").mean()

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(stl.trend.index, stl.trend.values, alpha=0.5, color="lightblue", label="Trend (daily)")
ax.plot(trend_monthly.index, trend_monthly.values, "o-", color="navy", linewidth=2, markersize=6,
        label="Trend (monthly mean)")
ax.set_xlabel("Date")
ax.set_ylabel("Revenue (£) — trend component only")
ax.set_title("Trend after removing weekly seasonality — annual pattern visible")
ax.legend()
plt.tight_layout()
plt.show()

# Year-over-year same-month comparison
print("Same-month trend value year-over-year:")
for month in [1, 6, 11]:
    v2024 = trend_monthly.loc[f"2024-{month:02d}"].mean()
    v2025 = trend_monthly.loc[f"2025-{month:02d}"].mean()
    print(f"  Month {month:02d}: 2024 £{v2024:,.0f} → 2025 £{v2025:,.0f}  (Δ £{v2025-v2024:+,.0f})")

## Step 5 — Inspect the residual

The residual is what's LEFT after removing trend and seasonal. It should look like noise. If you see patterns, you're missing a seasonality.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Residual over time
axes[0].plot(stl.resid.index, stl.resid.values, linewidth=0.7, color="seagreen")
axes[0].axhline(0, color="black", linewidth=0.5)
axes[0].set_ylabel("Residual (£)")
axes[0].set_title("Residual — what STL couldn't explain")

# Residual histogram
axes[1].hist(stl.resid.dropna(), bins=50, color="seagreen", edgecolor="white", alpha=0.85)
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_xlabel("Residual (£)")
axes[1].set_ylabel("Count")
axes[1].set_title("Residual distribution")

plt.tight_layout()
plt.show()

print(f"Residual mean: £{stl.resid.mean():.1f}  (should be near 0)")
print(f"Residual std:  £{stl.resid.std():.0f}")
print(f"Residual range: £{stl.resid.min():.0f} to £{stl.resid.max():.0f}")

### 💡 What you should notice

- **Mean near zero** ✓ — STL extracted the systematic patterns.
- **Roughly normal-looking distribution** ✓ — what's left is genuine random noise.
- **No obvious cyclical pattern in the time plot** ✓ — confirms we captured trend + weekly seasonal.

The residual *standard deviation* (≈ £380) tells you the scale of week-to-week noise on a normal (non-holiday) day. It's a useful sanity check, not a hard floor — on holiday windows the noise is much bigger, and on calm weeks a good model can beat the unconditional std by using features (e.g. lag_365) that capture some of what STL left as "residual".

## ⏸️ Pause and Predict

Before we set up cross-validation, predict:
- For a 90-day forecast, how should we SPLIT train vs test on this 731-day series?
- Should we use random k-fold? Why or why not?

> *Sample answers:*
> - Hold out the LAST 90 days as test. Use the first 641 days for training. Mimics deployment.
> - NEVER random k-fold — it would leak future data into past training folds. Use `TimeSeriesSplit`.

## Step 6 — Time-series cross-validation

`TimeSeriesSplit` gives you multiple train/test splits where the test set is ALWAYS later than the train set. We'll use 5 splits, each with a 60-day test window. This rolls forward through the data — like simulating real deployment 5 times.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5, test_size=60)

fig, ax = plt.subplots(figsize=(13, 4))

for split_idx, (train_idx, test_idx) in enumerate(tscv.split(y)):
    ax.scatter(y.index[train_idx], [split_idx] * len(train_idx),
                s=3, color="steelblue", label="Train" if split_idx == 0 else None)
    ax.scatter(y.index[test_idx], [split_idx] * len(test_idx),
                s=3, color="coral", label="Test" if split_idx == 0 else None)

ax.set_xlabel("Date")
ax.set_ylabel("Split #")
ax.set_yticks(range(5))
ax.set_title("TimeSeriesSplit — train always BEFORE test")
ax.legend(loc="lower right", markerscale=3)
plt.tight_layout()
plt.show()

print("Each row is one split. Train (blue) is always to the LEFT of test (coral).")
print(f"Test windows: 60 days each. Train windows grow as we move down (expanding).")

## Step 7 — A trivial baseline forecast + how to evaluate it

Let's forecast the last 60 days using the simplest possible method (Seasonal Naive — predict the value from 7 days earlier) and compute three standard error metrics.

In [ ]:
# Split: last 60 days as test
test_size = 60
y_train = y.iloc[:-test_size]
y_test  = y.iloc[-test_size:]

print(f"Train: {len(y_train)} days, from {y_train.index[0].date()} to {y_train.index[-1].date()}")
print(f"Test:  {len(y_test)} days, from {y_test.index[0].date()} to {y_test.index[-1].date()}")
print()

# Seasonal Naive — predict same-day-last-week for each test day
y_pred_snaive = pd.Series(index=y_test.index, dtype=float)
for date in y_test.index:
    last_week = date - pd.Timedelta(days=7)
    if last_week in y_train.index:
        y_pred_snaive.loc[date] = y_train.loc[last_week]
    else:
        # Fallback for the first week of test that doesn't have lag-7 in train
        y_pred_snaive.loc[date] = y_train.iloc[-7:].mean()

# Three standard metrics
mae  = mean_absolute_error(y_test, y_pred_snaive)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_snaive))
mape = (np.abs(y_test - y_pred_snaive) / y_test).mean() * 100

print("Seasonal Naive forecast (predict same-day-last-week):")
print(f"  MAE:   £{mae:,.0f}    (typical error magnitude)")
print(f"  RMSE:  £{rmse:,.0f}   (penalises large errors more)")
print(f"  MAPE:  {mape:.1f}%   (percentage error)")

### 💡 What this tells us

- **MAPE ~20%** — much higher than the "single-digit" target. Why?
- **Look at the test window:** it's the last 60 days of 2025 = **November and December** = the holiday spike.
- **Seasonal Naive uses lag-7** (same day last week), so it can't capture a RISING trend across weeks during the holiday season.
- **This is exactly the failure mode that classical forecasting (ETS) and ML forecasting (with lag features at multiple horizons) are designed to fix.** Tomorrow we'll see those methods do much better.

The bar to beat: **Seasonal Naive MAE ≈ £2,500 on this window**. Any "better" model needs to model the trend, not just copy last week.

## Visualise the forecast

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y_train.iloc[-90:].index, y_train.iloc[-90:].values, color="steelblue",
        linewidth=1.0, label="Train (last 90 days)")
ax.plot(y_test.index, y_test.values, "o-", color="darkgreen", markersize=4, linewidth=1.2,
        label="Actual (test)")
ax.plot(y_pred_snaive.index, y_pred_snaive.values, "s--", color="coral", markersize=4, linewidth=1.2,
        label="Seasonal Naive forecast")
ax.axvline(y_test.index[0], color="gray", linestyle="--", linewidth=1, alpha=0.7)
ax.text(y_test.index[0], ax.get_ylim()[1] * 0.95, " Forecast start",
        ha="left", fontsize=11, color="gray")
ax.set_xlabel("Date")
ax.set_ylabel("Revenue (£)")
ax.set_title(f"Seasonal Naive forecast — last 60 days (MAE: £{mae:,.0f}, MAPE: {mape:.1f}%)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## ✅ Section Summary

| Step | Output |
|---|---|
| **STL decomposition** | Trend (slow growth + annual pattern) + Seasonal (weekly cycle) + Residual (noise) |
| **Residual std ≈ £380** | Scale of unmodelled week-to-week noise on normal days (NOT a hard floor) |
| **TimeSeriesSplit** | 5 splits, each 60 days test, train always BEFORE test |
| **Seasonal Naive baseline on holiday window** | MAE ≈ £2,500, MAPE ≈ 20% — the bar the next notebooks must beat |

**Key insights:**
- **STL with period=7** captures weekly seasonality; the annual pattern goes into the trend.
- **MAE/RMSE/MAPE** are the three standard metrics; report all three or at least MAE + MAPE.
- **TimeSeriesSplit** is the only safe cross-validator for time series — k-fold leaks.
- **Seasonal Naive on the holiday window is poor (MAPE ~20%)** — because it can't model the November-December trend rise. NB 03's classical methods and NB 04's ML model will do better.

---
**Up next → Part 2:** Wednesday — Classical forecasting methods (Naive, Seasonal Naive, Exponential Smoothing). Build, compare, and pick a winner.
Open `03_classical_forecasting.ipynb`

---

## 🟡 Extension — self-study after class

*Skipping this section will not affect your understanding of later lessons. Come back to it when you have time and want to go deeper.*

## Extension 1 — STL with period = 365 (annual seasonality only)

What if we use period=365 instead of period=7? The decomposition will allocate the ANNUAL cycle to seasonal and the WEEKLY noise will spill into the residual. Visible failure mode.

In [ ]:
stl_annual = STL(y, period=365, robust=True).fit()

fig, axes = plt.subplots(2, 1, figsize=(13, 6))
axes[0].plot(stl_annual.seasonal.index, stl_annual.seasonal.values, color="steelblue", linewidth=0.8)
axes[0].set_title("Seasonal component with period=365 (captures annual cycle)")
axes[0].set_ylabel("Seasonal (£)")
axes[0].axhline(0, color="black", linewidth=0.5)

axes[1].plot(stl_annual.resid.index, stl_annual.resid.values, color="seagreen", linewidth=0.7)
axes[1].set_title("Residual with period=365 (weekly cycle leaks into here)")
axes[1].set_ylabel("Residual (£)")
axes[1].axhline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

print(f"Residual std with period=7:   £{stl.resid.std():.0f}")
print(f"Residual std with period=365: £{stl_annual.resid.std():.0f}  (HIGHER — weekly cycle leaked)")
print()
print("Lesson: pick period to match the DOMINANT seasonality. Multi-season needs MSTL or lag features.")

## Extension 2 — Multi-seasonal decomposition (MSTL)

For series with BOTH weekly AND annual seasonality, statsmodels has MSTL — Multiple Seasonal-Trend decomposition. It runs STL iteratively, peeling off one seasonality at a time.

In [ ]:
from statsmodels.tsa.seasonal import MSTL

mstl = MSTL(y, periods=(7, 365)).fit()

# Plot the components
fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)
axes[0].plot(y.index, y.values, color="black", linewidth=0.5)
axes[0].set_title("Observed")
axes[1].plot(mstl.trend.index, mstl.trend.values, color="navy", linewidth=1.2)
axes[1].set_title("Trend")
axes[2].plot(mstl.seasonal.index, mstl.seasonal.iloc[:, 0], color="steelblue", linewidth=0.6)
axes[2].set_title("Seasonal — period 7 (weekly)")
axes[3].plot(mstl.seasonal.index, mstl.seasonal.iloc[:, 1], color="coral", linewidth=0.6)
axes[3].set_title("Seasonal — period 365 (annual)")
plt.tight_layout()
plt.show()

print(f"MSTL trend (with both seasonalities removed): smoother than STL trend")
print(f"Annual seasonal range: £{mstl.seasonal.iloc[:, 1].max() - mstl.seasonal.iloc[:, 1].min():.0f}")
print(f"Weekly seasonal range: £{mstl.seasonal.iloc[:, 0].max() - mstl.seasonal.iloc[:, 0].min():.0f}")

## Extension 3 — Read the autocorrelation function (ACF)

ACF plots correlation of the series with its own lagged version. Spikes at lag 7, 14, 21 → weekly seasonality. Spike at lag 365 → annual.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

fig, axes = plt.subplots(2, 1, figsize=(13, 7))
plot_acf(y, lags=30, ax=axes[0], title="ACF — short range (look for weekly spikes at lag 7, 14, 21, 28)")
plot_acf(y, lags=400, ax=axes[1], title="ACF — long range (look for annual spike near lag 365)")
plt.tight_layout()
plt.show()
print("ACF tells you which lag features to add when building the ML forecaster on Thursday.")